# Load exploration: Supabase (PostgreSQL)

Findings that shaped `src/load.py`:

- **Target schema:** `raw` (not `public`, which Supabase exposes via its REST API).
- **Types:** int64 → bigint, str → text (postal codes keep leading zeros), datetime64 → timestamp, bool → boolean.
- **Speed:** `to_sql(method="multi")` loaded erp_orders (138k × 46) in 481 s; PostgreSQL `COPY` took 134 s. Remaining bottleneck is home upload bandwidth (~0.36 MB/s).
- **Timeout:** Supabase default `statement_timeout` is 2 min; raised per transaction with `SET LOCAL`.
- **Atomicity:** a COPY that failed at row 132,238 was rolled back and the previous table stayed intact.
- **CSV NULL risk:** COPY reads empty values as NULL, so text columns are checked for empty strings first (0 found).
- **Reconciliation:** row count, SUM(net_sales) = 177,134,263.74 and NULL counts match between Pandas and Supabase.

In [1]:
import os
from dotenv import load_dotenv

# Read variables from the .env file in the project root
load_dotenv()

# Print only non-secret values; never print the password itself
print("Host:", os.getenv("SUPABASE_DB_HOST"))
print("Port:", os.getenv("SUPABASE_DB_PORT"))
print("User:", os.getenv("SUPABASE_DB_USER"))
print("Password loaded:", bool(os.getenv("SUPABASE_DB_PASSWORD")))

Host: aws-0-ap-southeast-1.pooler.supabase.com
Port: 5432
User: postgres.gfekazmvtknqfmokwutm
Password loaded: True


In [2]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Build the URL from parts so special characters in the password are handled safely
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("SUPABASE_DB_USER"),
    password=os.getenv("SUPABASE_DB_PASSWORD"),
    host=os.getenv("SUPABASE_DB_HOST"),
    port=int(os.getenv("SUPABASE_DB_PORT")),
    database=os.getenv("SUPABASE_DB_NAME"),
)

# sslmode=require encrypts traffic between the laptop and Supabase
engine = create_engine(db_url, connect_args={"sslmode": "require"})

with engine.connect() as conn:
    row = conn.execute(text("SELECT version(), current_user, current_database()")).fetchone()
    print(row)

engine.dispose()

('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit', 'postgres', 'postgres')


In [3]:
# Create a dedicated schema for source-shaped data (not exposed via Supabase REST API)
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

print("Schema 'raw' is ready")

Schema 'raw' is ready


In [4]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("data/processed")

web_users = pd.read_parquet(PROCESSED_DIR / "web_users.parquet")

print(web_users.shape)
print(web_users.dtypes)

(208, 13)
user_id               int64
first_name              str
last_name               str
age                   int64
gender                  str
role                    str
city                    str
state                   str
state_code              str
postal_code             str
country                 str
company_department      str
job_title               str
dtype: object


In [5]:
# Write the DataFrame into raw.web_users (full refresh)
rows_written = web_users.to_sql(
    name="web_users",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)

print("Rows written:", rows_written)

Rows written: 208


In [6]:
# Verify row count and column types directly from the database, not from Pandas
with engine.connect() as conn:
    db_count = conn.execute(text("SELECT COUNT(*) FROM raw.web_users")).scalar()
    columns = conn.execute(text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw' AND table_name = 'web_users'
        ORDER BY ordinal_position
    """)).fetchall()

print("Rows in Pandas  :", len(web_users))
print("Rows in Supabase:", db_count)
for col in columns:
    print(col)

Rows in Pandas  : 208
Rows in Supabase: 208
('user_id', 'bigint')
('first_name', 'text')
('last_name', 'text')
('age', 'bigint')
('gender', 'text')
('role', 'text')
('city', 'text')
('state', 'text')
('state_code', 'text')
('postal_code', 'text')
('country', 'text')
('company_department', 'text')
('job_title', 'text')


In [7]:
erp_orders = pd.read_parquet(PROCESSED_DIR / "erp_orders.parquet")

print(erp_orders.shape)
print(erp_orders.dtypes.value_counts())   # how many columns of each type
print()
print(erp_orders.dtypes.to_string())      # full list, one line per column

(138116, 46)
str               26
float64           12
int64              5
datetime64[us]     2
bool               1
Name: count, dtype: int64

order_id                               str
order_date                  datetime64[us]
order_datetime              datetime64[us]
order_status                           str
sales_channel                          str
customer_id                            str
customer_name                          str
customer_age                         int64
gender                                 str
customer_segment                       str
customer_type                          str
customer_city                          str
customer_state                         str
customer_country                       str
region                                 str
customer_postal_code                   str
payment_method                         str
payment_status                         str
currency                               str
shipping_method                       

In [8]:
import time

start = time.perf_counter()

rows_written = erp_orders.to_sql(
    name="erp_orders",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)

elapsed = time.perf_counter() - start
print(f"Rows written: {rows_written}")
print(f"Load time   : {elapsed:.1f} seconds")

Rows written: 138116
Load time   : 480.9 seconds


In [15]:
with engine.connect() as conn:
    checks = conn.execute(text("""
        SELECT
            COUNT(*)                         AS row_count,
            ROUND(SUM(net_sales)::numeric, 2) AS total_net_sales,
            COUNT(*) - COUNT(delivery_days)  AS null_delivery_days,
            MIN(order_date)                  AS min_date,
            MAX(order_date)                  AS max_date
        FROM raw.erp_orders
    """)).mappings().one()

    types = conn.execute(text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'raw' AND table_name = 'erp_orders'
          AND column_name IN ('order_date', 'order_datetime', 'is_repeat_customer',
                              'net_sales', 'customer_postal_code', 'quantity')
        ORDER BY column_name
    """)).fetchall()

print("Supabase:", dict(checks))
print("Pandas  :", {
    "row_count": len(erp_orders),
    "total_net_sales": round(erp_orders["net_sales"].sum(), 2),
    "null_delivery_days": int(erp_orders["delivery_days"].isna().sum()),
    "min_date": erp_orders["order_date"].min(),
    "max_date": erp_orders["order_date"].max(),
})
print()
for t in types:
    print(t)

Supabase: {'row_count': 138116, 'total_net_sales': Decimal('177134263.74'), 'null_delivery_days': 24557, 'min_date': datetime.datetime(2021, 1, 1, 0, 0), 'max_date': datetime.datetime(2025, 12, 31, 0, 0)}
Pandas  : {'row_count': 138116, 'total_net_sales': np.float64(177134263.74), 'null_delivery_days': 24557, 'min_date': Timestamp('2021-01-01 00:00:00'), 'max_date': Timestamp('2025-12-31 00:00:00')}

('customer_postal_code', 'text')
('is_repeat_customer', 'boolean')
('net_sales', 'double precision')
('order_date', 'timestamp without time zone')
('order_datetime', 'timestamp without time zone')
('quantity', 'bigint')


In [10]:
# Empty strings would silently become NULL during CSV-based COPY, so confirm there are none
text_cols = erp_orders.select_dtypes(include="str").columns
empty_strings = int((erp_orders[text_cols] == "").sum().sum())
print("Empty strings in text columns:", empty_strings)

Empty strings in text columns: 0


In [12]:
with engine.connect() as conn:
    timeout = conn.execute(text("SHOW statement_timeout")).scalar()
    existing_rows = conn.execute(text("SELECT COUNT(*) FROM raw.erp_orders")).scalar()

print("Default statement_timeout:", timeout)
print("Rows still in raw.erp_orders:", existing_rows)

Default statement_timeout: 2min
Rows still in raw.erp_orders: 138116


In [13]:
def copy_load(df, table_name, schema="raw"):
    """Replace schema.table_name with df using PostgreSQL COPY (fast bulk load)."""
    columns = ", ".join(f'"{c}"' for c in df.columns)

    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    payload_mb = buffer.tell() / 1024**2   # approximate size of data sent over the network
    buffer.seek(0)

    with engine.begin() as conn:
        # Raise the timeout for this transaction only; it resets automatically afterwards
        conn.execute(text("SET LOCAL statement_timeout = '10min'"))
        df.head(0).to_sql(table_name, conn, schema=schema, if_exists="replace", index=False)
        with conn.connection.cursor() as cur:
            cur.copy_expert(
                f"COPY {schema}.{table_name} ({columns}) FROM STDIN WITH (FORMAT csv)",
                buffer,
            )
    return payload_mb

start = time.perf_counter()
size_mb = copy_load(erp_orders, "erp_orders")
print(f"Payload size  : {size_mb:.1f} MB")
print(f"COPY load time: {time.perf_counter() - start:.1f} seconds")

Payload size  : 48.6 MB
COPY load time: 133.6 seconds


In [16]:
with engine.connect() as conn:
    sizes = conn.execute(text("""
        SELECT
            pg_size_pretty(pg_database_size(current_database()))   AS whole_database,
            pg_size_pretty(pg_total_relation_size('raw.erp_orders')) AS erp_orders,
            pg_size_pretty(pg_total_relation_size('raw.web_users'))  AS web_users
    """)).mappings().one()

print(dict(sizes))

{'whole_database': '71 MB', 'erp_orders': '60 MB', 'web_users': '64 kB'}
